# 18.08 - R3D-18 transfer learning

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** Frozen-head versus layer4-unfrozen R3D-18 comparison.

Configure two exact R3D-18 transfer strategies, train each on the same tiny video split, and compare validation Macro-F1 and runtime without requiring a checkpoint download.

## Core Ideas

Head-only training is cheap and preserves backbone features. Unfreezing `layer4` gives more adaptability but increases compute and overfitting risk. Replace `model.fc`, control `requires_grad`, use smaller learning rates for backbone parameters, and keep validation in evaluation mode.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import f1_score
from torchvision.models.video import r3d_18

SEED = 18
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared Video Split

Nine clips form three balanced classes. Six clips train and three disjoint clips validate. Inputs already use `[B,C,T,H,W]`.

In [ ]:
all_clips = torch.zeros((9, 3, 4, 32, 32), dtype=torch.float32)
all_labels = torch.arange(3).repeat_interleave(3)
for index, label in enumerate(all_labels):
    all_clips[index, label, :, 6 + index:18 + index, 8:20] = 1.0
train_indices = torch.tensor([0,1,3,4,6,7])
validation_indices = torch.tensor([2,5,8])
train_clips, train_labels = all_clips[train_indices], all_labels[train_indices]
validation_clips, validation_labels = all_clips[validation_indices], all_labels[validation_indices]
print("train/validation support:", torch.bincount(train_labels).tolist(), torch.bincount(validation_labels).tolist())

## Exercise 18-A: Configure transfer parameters

Start from the exact architecture, freeze all parameters, replace the head, and optionally unfreeze `layer4`.

**Return structure — `configure_r3d_transfer`:** A `VideoResNet` on `device` with three output classes. `model.fc` is trainable; `layer4` is trainable only when `unfreeze_layer4=True`.

In [ ]:
def configure_r3d_transfer(num_classes=3, unfreeze_layer4=False, device=DEVICE):
    model = r3d_18(weights=None)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.fc = nn.Linear(model.fc.in_features, int(num_classes))
    if unfreeze_layer4:
        for parameter in model.layer4.parameters():
            parameter.requires_grad_(True)
    return model.to(device)


# Smoke check: compare trainable parameter counts.
frozen_model = configure_r3d_transfer(unfreeze_layer4=False)
partial_model = configure_r3d_transfer(unfreeze_layer4=True)
print(sum(p.numel() for p in frozen_model.parameters() if p.requires_grad), sum(p.numel() for p in partial_model.parameters() if p.requires_grad))

## Exercise 18-B: Create learning-rate groups

Use a smaller learning rate for unfrozen backbone parameters than for the new classifier head.

**Return structure — `transfer_parameter_groups`:** A `list[dict]` accepted by a PyTorch optimizer. Every dictionary has `params` (non-empty iterable) and float `lr`; length is one for head-only and two when layer4 is trainable.

In [ ]:
def transfer_parameter_groups(model, head_lr=0.01, backbone_lr=0.001):
    head_parameters = [parameter for parameter in model.fc.parameters() if parameter.requires_grad]
    backbone_parameters = [parameter for name, parameter in model.named_parameters() if not name.startswith("fc.") and parameter.requires_grad]
    groups = [{"params": head_parameters, "lr": float(head_lr)}]
    if backbone_parameters:
        groups.append({"params": backbone_parameters, "lr": float(backbone_lr)})
    return groups


# Smoke check: inspect both optimizer configurations.
frozen_groups = transfer_parameter_groups(frozen_model)
partial_groups = transfer_parameter_groups(partial_model)
print([group["lr"] for group in frozen_groups], [group["lr"] for group in partial_groups])

## Exercise 18-C: Train and evaluate one strategy

Train on all six training clips, evaluate the same three validation clips, and report runtime and Macro-F1.

**Return structure — `train_r3d_strategy`:** A dictionary with Python floats `train_loss`, `macro_f1`, and `runtime_seconds`, integer `train_size` and `validation_size`, `validation_support` as `list[int]`, and predictions as `list[int]`.

In [ ]:
def train_r3d_strategy(model, train_data, train_targets, validation_data, validation_targets, epochs=1, device=DEVICE):
    optimizer = torch.optim.SGD(transfer_parameter_groups(model), momentum=0.9)
    start = time.perf_counter(); last_loss = 0.0
    for _ in range(epochs):
        model.train(); optimizer.zero_grad(); logits = model(train_data.to(device)); loss = nn.functional.cross_entropy(logits, train_targets.to(device)); loss.backward(); optimizer.step(); last_loss = float(loss.detach().cpu())
    model.eval()
    with torch.inference_mode(): predictions = model(validation_data.to(device)).argmax(dim=1).cpu()
    return {"train_loss": last_loss, "macro_f1": float(f1_score(validation_targets.numpy(), predictions.numpy(), average="macro")), "runtime_seconds": time.perf_counter() - start, "train_size": len(train_targets), "validation_size": len(validation_targets), "validation_support": torch.bincount(validation_targets, minlength=3).tolist(), "predictions": predictions.tolist()}


# Smoke check and complete split comparison.
frozen_result = train_r3d_strategy(frozen_model, train_clips, train_labels, validation_clips, validation_labels)
partial_result = train_r3d_strategy(partial_model, train_clips, train_labels, validation_clips, validation_labels)
print("frozen/partial:", frozen_result, partial_result)

## Exercise 18-D: Build a controlled comparison

Do not require either stochastic strategy to win; expose its change from the head-only baseline.

**Return structure — `r3d_transfer_table`:** A two-row DataFrame with columns `strategy`, `train_size`, `validation_size`, `validation_support`, `macro_f1`, `runtime_seconds`, and `delta_macro_f1`.

In [ ]:
def r3d_transfer_table(frozen, partial):
    rows = []
    for name, result in [("head_only", frozen), ("unfreeze_layer4", partial)]:
        rows.append({"strategy": name, **{key: result[key] for key in ["train_size", "validation_size", "validation_support", "macro_f1", "runtime_seconds"]}})
    table = pd.DataFrame(rows); table["delta_macro_f1"] = table["macro_f1"] - float(table.iloc[0]["macro_f1"]); return table


# Smoke check: print aligned evidence.
r3d_transfer_evidence = r3d_transfer_table(frozen_result, partial_result)
print(r3d_transfer_evidence.to_string(index=False))

## Test Cases

**Return structure — `run_day18_tests`:** Returns `None`; assertions and `Day 18 tests passed` communicate success.

In [ ]:
def run_day18_tests():
    assert set(train_indices.tolist()).isdisjoint(set(validation_indices.tolist()))
    assert len(frozen_groups) == 1 and len(partial_groups) == 2
    assert frozen_model.fc.out_features == partial_model.fc.out_features == 3
    assert frozen_result["validation_support"] == partial_result["validation_support"] == [1, 1, 1]
    assert frozen_result["validation_size"] == partial_result["validation_size"] == 3
    assert r3d_transfer_evidence.shape == (2, 7)
    assert float(r3d_transfer_evidence.iloc[0]["delta_macro_f1"]) == 0.0
    print("Day 18 tests passed")


run_day18_tests()

## Day 18 Checklist

- [ ] Replace the classifier head for target classes.
- [ ] Freeze or unfreeze explicit backbone stages.
- [ ] Use a smaller backbone learning rate.
- [ ] Compare identical train and validation splits.
- [ ] Run the test cases.